In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F

from transformers import VisionEncoderDecoderModel, AutoProcessor

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer

model_name = "nlpconnect/vit-gpt2-image-captioning"

model = VisionEncoderDecoderModel.from_pretrained(
    model_name,
    attn_implementation="eager"  
)

image_processor = ViTImageProcessor.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Enable attentions
model.config.output_attentions = True
model.config.decoder.output_attentions = True

model.to(device)
model.eval()

In [ ]:
def normalize_distribution(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float64)
    x = np.maximum(x, eps)
    x = x / np.sum(x)
    return x

def kl_divergence(p, q, eps=1e-12):
    p = normalize_distribution(p, eps)
    q = normalize_distribution(q, eps)
    return np.sum(p * np.log((p + eps) / (q + eps)))

def js_divergence(p, q, eps=1e-12):
    p = normalize_distribution(p, eps)
    q = normalize_distribution(q, eps)
    m = 0.5 * (p + q)
    return 0.5 * kl_divergence(p, m, eps) + 0.5 * kl_divergence(q, m, eps)

In [ ]:
base_dir = "Enter the path"
image_path = os.path.join(base_dir, "images", "val", "COCO_val2014_000000000133.jpg")

image = Image.open(image_path).convert("RGB")

pixel_values = image_processor(images=image, return_tensors="pt").pixel_values
pixel_values = pixel_values.to(device)

with torch.no_grad():
    outputs = model.generate(
        pixel_values=pixel_values,
        output_attentions=True,
        return_dict_in_generate=True
    )

generated_ids = outputs.sequences
caption = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print("Caption:", caption)

In [ ]:
with torch.no_grad():
    forward_outputs = model(
        pixel_values=pixel_values,
        labels=generated_ids,
        output_attentions=True,
        return_dict=True
    )

cross_attentions = forward_outputs.cross_attentions

print("Number of decoder layers:", len(cross_attentions))
print("Cross-attention shape (layer 0):", cross_attentions[0].shape)

In [ ]:
# Last decoder layer
last_layer_attention = cross_attentions[-1]  # (batch, heads, tokens, patches)

# Remove batch dimension
last_layer_attention = last_layer_attention[0]  # (heads, tokens, patches)

# Average over heads
attention_avg = last_layer_attention.mean(dim=0)  # (tokens, patches)

num_tokens, num_patches = attention_avg.shape

print("Raw patch count:", num_patches)

attention_avg = attention_avg[:, 1:]  # remove CLS token

num_tokens, num_patches = attention_avg.shape

print("Patch count after CLS removal:", num_patches)

grid_size = int(math.sqrt(num_patches))
assert grid_size * grid_size == num_patches, "Patches do not form square grid"

token_maps = []

for t in range(num_tokens):
    token_attention = attention_avg[t]  # (196,)
    token_map = token_attention.view(grid_size, grid_size).detach().cpu().numpy()
    token_map = normalize_distribution(token_map)
    token_maps.append(token_map)

token_maps = np.array(token_maps)

print("Final token maps shape:", token_maps.shape)

In [ ]:
saliency_path = os.path.join(base_dir, "maps", "val", "COCO_val2014_000000000133.png")

saliency = Image.open(saliency_path).convert("L")
saliency = saliency.resize((grid_size, grid_size), Image.BILINEAR)

saliency_np = np.array(saliency, dtype=np.float64)
saliency_np = normalize_distribution(saliency_np)

print("Saliency map shape:", saliency_np.shape)

In [ ]:
token_JS = []
token_KL = []

for token_map in token_maps:
    js = js_divergence(token_map, saliency_np)
    kl = kl_divergence(token_map, saliency_np)
    token_JS.append(js)
    token_KL.append(kl)

print("Average JS:", np.mean(token_JS))
print("Average KL:", np.mean(token_KL))

In [ ]:
def process_image_vit_gpt2(image_path, saliency_path):
    image = Image.open(image_path).convert("RGB")

    # Image → pixel values
    pixel_values = image_processor(images=image, return_tensors="pt").pixel_values
    pixel_values = pixel_values.to(device)

    # Generate caption
    with torch.no_grad():
        generated = model.generate(
            pixel_values=pixel_values,
            return_dict_in_generate=True
        )

    generated_ids = generated.sequences
    caption = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    # Forward pass for cross-attention
    with torch.no_grad():
        outputs = model(
            pixel_values=pixel_values,
            decoder_input_ids=generated_ids,
            output_attentions=True,
            return_dict=True
        )

    cross_attentions = outputs.cross_attentions

    if cross_attentions is None:
        raise RuntimeError("Cross attentions not returned. Check eager attention setup.")

    # Last decoder layer
    last_layer_attention = cross_attentions[-1][0]  # (heads, tokens, patches)

    # Average over heads
    attention_avg = last_layer_attention.mean(dim=0)  # (tokens, patches)

    # Remove CLS patch
    attention_avg = attention_avg[:, 1:]

    num_tokens, num_patches = attention_avg.shape
    grid_size = int(math.sqrt(num_patches))

    # Load saliency
    saliency = Image.open(saliency_path).convert("L")
    saliency = saliency.resize((grid_size, grid_size), Image.BILINEAR)
    saliency_np = normalize_distribution(np.array(saliency, dtype=np.float64))

    token_JS = []
    token_KL = []

    for t in range(num_tokens):
        token_map = attention_avg[t].view(grid_size, grid_size).detach().cpu().numpy()
        token_map = normalize_distribution(token_map)

        token_JS.append(js_divergence(token_map, saliency_np))
        token_KL.append(kl_divergence(token_map, saliency_np))

    return {
        "caption": caption,
        "num_tokens": num_tokens,
        "avg_JS": float(np.mean(token_JS)),
        "avg_KL": float(np.mean(token_KL)),
        "token_JS_list": token_JS,
        "token_KL_list": token_KL
    }

In [ ]:
image_dir = os.path.join(base_dir, "images", "val")
saliency_dir = os.path.join(base_dir, "maps", "val")

results = []

image_files = [f for f in os.listdir(image_dir) if f.endswith(".jpg")]

for img_file in tqdm(image_files):
    image_path = os.path.join(image_dir, img_file)
    saliency_file = img_file.replace(".jpg", ".png")
    saliency_path = os.path.join(saliency_dir, saliency_file)

    if not os.path.exists(saliency_path):
        continue

    result = process_image_vit_gpt2(image_path, saliency_path)
    result["image_id"] = img_file.replace(".jpg", "")
    results.append(result)

In [ ]:
df = pd.DataFrame(results)

df = df[[
    "image_id",
    "caption",
    "num_tokens",
    "avg_JS",
    "avg_KL",
    "token_JS_list",
    "token_KL_list"
]]

print(df.head())
print("Total processed images:", len(df))

In [ ]:
output_path = os.path.join(base_dir, "vit_gpt2_attention_alignment_results.csv")

df.to_csv(output_path, index=False)

print("CSV saved to:", output_path)

In [ ]:
print("Dataset mean JS:", df["avg_JS"].mean())
print("Dataset std JS:", df["avg_JS"].std())

print("Dataset mean KL:", df["avg_KL"].mean())
print("Dataset std KL:", df["avg_KL"].std())